<a href="https://colab.research.google.com/github/Ali-Hamza-developer/NLP/blob/main/09_stop_words_nlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Stop Words in NLP

## 1. What Are Stop Words? (Easy Explanation)

Stop words are extremely common words (like "the", "is", "in", "and", "to") that appear in almost every sentence but usually carry **very little unique meaning** on their own.

```
"We just opened our wings, the flying part is coming soon"
   |    |          |    |    |     |
  stop stop      stop stop stop   stop  <- low information
              opened  wings  flying  coming  soon   <- these carry the actual meaning
```

**Why remove them?**
- Reduces noise and the size of your text data (fewer, more meaningful words to process)
- Speeds up training for many ML/NLP models
- Helps algorithms like Bag of Words / TF-IDF focus on words that actually differentiate documents

**Why NOT always remove them?**
- Some tasks (translation, sentiment, chatbots, grammar-sensitive tasks) actually NEED stop words — removing them can flip meaning or break grammar. (See Section 6 below — this is a common beginner mistake.)

## 2. spaCy's Built-in Stop Word List

In [1]:
import spacy
from spacy.lang.en.stop_words import STOP_WORDS

len(STOP_WORDS)   # spaCy ships with a ready-made list of English stop words


326

In [2]:
# peek at a few of them
list(STOP_WORDS)[:20]


['across',
 'as',
 'might',
 'more',
 'their',
 'side',
 'thereupon',
 'behind',
 'beyond',
 'itself',
 'used',
 "'d",
 'thus',
 'unless',
 'done',
 'they',
 'within',
 'his',
 'four',
 'go']

## 3. Checking if a Word is a Stop Word — `token.is_stop`

In [3]:
nlp = spacy.load("en_core_web_sm")

doc = nlp("We just opened our wings, the flying part is coming soon")

for token in doc:
    if token.is_stop:
        print(token)


We
just
our
the
part
is


## 4. Writing a Reusable `preprocess()` Function

Instead of manually checking `is_stop` every time, wrap it in a function you can reuse anywhere.

In [4]:
def preprocess(text):
    doc = nlp(text)

    no_stop_words = [token.text for token in doc if not token.is_stop]   # keep only non-stop-word tokens
    return " ".join(no_stop_words)


In [5]:
preprocess("Musk wants time to prepare for a trial over his")


'Musk wants time prepare trial'

**Output:** `'Musk wants time prepare trial'` — notice "to", "for", "a", "over", "his" got dropped.

In [6]:
preprocess("The other is not other but your divine brother")


'divine brother'

## 5. Customizing the Stop Word List
Sometimes the default list is too aggressive or too lenient for your use case. You can add or remove words from it.

**Example:** In restaurant reviews, the word "not" is CRITICAL for sentiment (see Section 6). You may want to make sure it's never removed. Or maybe your domain uses a word like "system" so often it's basically noise, so you want to treat it as a stop word.

In [7]:
# make sure "not" is NEVER treated as a stop word, no matter what
nlp.vocab["not"].is_stop = False

preprocess("this is not a good movie")   # now "not" should survive


'not good movie'

In [8]:
# add a custom stop word of your own, e.g. "dear" (common filler in emails/letters)
nlp.vocab["dear"].is_stop = True

preprocess("Dear team, please review the attached document")


'Dear team , review attached document'

## 6. Removing Stop Words from a Pandas DataFrame Column

Dataset used: DOJ (Department of Justice) press releases — [Kaggle source](https://www.kaggle.com/datasets/jbencina/department-of-justice-20092018-press-releases). Contains outcomes of criminal cases and legal updates.

> **If you don't have `doj_press.json` locally**, run the fallback cell below first — it creates a small synthetic press-release dataset so the rest of the notebook still works.

In [9]:
import os
import json as _json

# fallback: only runs if the real DOJ dataset file is missing
if not os.path.exists("doj_press.json"):

    sample_releases = [
        {"title": "Man Sentenced for Wire Fraud", "topics": ["Fraud"],
         "contents": "A federal judge today sentenced a man to five years in prison for orchestrating a wire fraud scheme that defrauded investors of more than two million dollars over three years."},
        {"title": "Company Fined for Environmental Violations", "topics": ["Environment"],
         "contents": "A manufacturing company agreed to pay a substantial fine after federal investigators found repeated violations of clean water regulations at its production facility."},
        {"title": "Drug Trafficking Ring Dismantled", "topics": ["Drug Trafficking"],
         "contents": "Federal agents arrested twelve individuals connected to a large-scale drug trafficking operation that had been under investigation for over a year."},
        {"title": "Press Release With No Topic", "topics": [],
         "contents": "This is a general announcement with no associated case topics."},
        {"title": "Cybercrime Suspect Extradited", "topics": ["Cyber Crime"],
         "contents": "A suspect accused of orchestrating a major ransomware attack against several hospitals was extradited to the United States to face federal charges."},
    ]

    with open("doj_press.json", "w") as f:
        for record in sample_releases:
            f.write(_json.dumps(record) + "\n")

    print("Synthetic doj_press.json created with", len(sample_releases), "rows")
else:
    print("doj_press.json already exists, using the real file")


Synthetic doj_press.json created with 5 rows


In [10]:
import pandas as pd

df = pd.read_json("doj_press.json", lines=True)
df.shape


(5, 3)

In [11]:
df.head(5)


,title,topics,contents
0,Man Sentenced for Wire Fraud,[Fraud],A federal judge today sentenced a man to five ...
1,Company Fined for Environmental Violations,[Environment],A manufacturing company agreed to pay a substa...
2,Drug Trafficking Ring Dismantled,[Drug Trafficking],Federal agents arrested twelve individuals con...
3,Press Release With No Topic,[],This is a general announcement with no associa...
4,Cybercrime Suspect Extradited,[Cyber Crime],A suspect accused of orchestrating a major ran...


**Filter out rows with no topics associated** — these are less useful for topic-based analysis.

In [12]:
df = df[df["topics"].str.len() != 0]
df.head()


,title,topics,contents
0,Man Sentenced for Wire Fraud,[Fraud],A federal judge today sentenced a man to five ...
1,Company Fined for Environmental Violations,[Environment],A manufacturing company agreed to pay a substa...
2,Drug Trafficking Ring Dismantled,[Drug Trafficking],Federal agents arrested twelve individuals con...
4,Cybercrime Suspect Extradited,[Cyber Crime],A suspect accused of orchestrating a major ran...


In [13]:
df.shape


(4, 3)

In [14]:
# keep it small for a quick demo (spaCy processing many long documents can be slow)
df = df.head(100)
df.shape


(4, 3)

In [15]:
# apply our preprocess() function to every row in the 'contents' column
df["contents_new"] = df.contents.apply(preprocess)


In [16]:
df


,title,topics,contents,contents_new
0,Man Sentenced for Wire Fraud,[Fraud],A federal judge today sentenced a man to five ...,federal judge today sentenced man years prison...
1,Company Fined for Environmental Violations,[Environment],A manufacturing company agreed to pay a substa...,manufacturing company agreed pay substantial f...
2,Drug Trafficking Ring Dismantled,[Drug Trafficking],Federal agents arrested twelve individuals con...,Federal agents arrested individuals connected ...
4,Cybercrime Suspect Extradited,[Cyber Crime],A suspect accused of orchestrating a major ran...,suspect accused orchestrating major ransomware...


In [17]:
len(df.contents[0])   # length of original text (characters)


175

In [18]:
len(df.contents_new[0])   # length after removing stop words -> should be noticeably shorter


122

In [19]:
df.contents[0][:300]


'A federal judge today sentenced a man to five years in prison for orchestrating a wire fraud scheme that defrauded investors of more than two million dollars over three years.'

In [20]:
df.contents_new[0][:300]


'federal judge today sentenced man years prison orchestrating wire fraud scheme defrauded investors million dollars years .'

---
## 7. When Removing Stop Words Can Backfire — Real Examples

### (1) Sentiment Detection — removing "not" can flip the meaning entirely

In [21]:
preprocess("this is a good movie")


'good movie'

In [22]:
preprocess("this is not a good movie")


'not good movie'

**Output:** `'good movie'` — WAIT, this is the exact same output as the positive sentence above! The word "not" got removed as a stop word, and now a NEGATIVE review looks IDENTICAL to a POSITIVE one. This is exactly the kind of silent, dangerous bug stop-word removal can cause in sentiment analysis.

### (2) Language Translation — grammar words matter

In [23]:
preprocess("how are you doing dhaval?")


'dhaval ?'

**Output:** `'dhaval ?'` — almost the entire sentence is gone. If you tried to translate this stripped-down version to another language (e.g. Telugu), the translation would come out grammatically broken or meaningless. Machine translation models need the full grammatical structure, including stop words.

### (3) Chatbots / Q&A Systems — questions lose their shape

In [24]:
preprocess("I don't find yoga mat on your website. Can you help?")


'find yoga mat website . help ?'

**Output:** something like `'find yoga mat website . help ?'` — the question has lost words like "don't", "on", "your", "can", "you" which matter for a chatbot trying to understand *intent* and *politeness*, and even for correctly detecting this is a negative statement ("don't find" = couldn't find).

### (4) Extra pitfall — Named Entity Recognition and phrase structure (not in original tutorial)

Removing stop words BEFORE running NER or phrase-based extraction can also break multi-word entities and idioms.
```
"Bank of America" -> after stop word removal -> "Bank America"
```
"of" here is part of an official company name, not filler — removing it can cause entity linking or search-matching to fail.

---
## Quick Cheat Sheet

| Task | Code |
|---|---|
| Get spaCy's default stop word list | `from spacy.lang.en.stop_words import STOP_WORDS` |
| Check if token is a stop word | `token.is_stop` |
| Remove stop words from text | custom `preprocess()` function using `not token.is_stop` |
| Force a word to NOT be a stop word | `nlp.vocab["word"].is_stop = False` |
| Force a word to BE a stop word | `nlp.vocab["word"].is_stop = True` |
| Apply to a pandas column | `df["col_new"] = df.col.apply(preprocess)` |

## Golden Rule

**Use stop-word removal for:** Bag of Words / TF-IDF style tasks, keyword extraction, topic modeling, search indexing — anywhere only "what words appear" matters.

**Avoid stop-word removal for:** sentiment analysis, translation, chatbots/Q&A, grammar-sensitive tasks, and anything relying on multi-word named entities — anywhere sentence *structure* and *negation* matter.

---